In [3]:
import xarray as xr 
from anemoi.datasets import open_dataset
import numpy as np
import yaml
import os 
from pathlib import Path

### Variables

In [4]:
ds = xr.open_zarr("Cerra_boz.zarr")

print(ds.dims)
print(ds.coords)

for name in ("level", "levelist", "isobaricInhPa", "plev", "pressure"):
    if name in ds.coords:
        print("Pressure coord:", name, ds.coords[name].values)
for var in ("u","v","t","z","r"):
    if var in ds:
        print(var, ds[var].dims)

FrozenMappingWarningOnValuesAccess({'time': 7544, 'y': 157, 'x': 211, 'level': 9})
Coordinates:
    latitude    (y, x) float64 265kB dask.array<chunksize=(157, 211), meta=np.ndarray>
  * level       (level) float64 72B 1e+03 950.0 900.0 ... 700.0 600.0 500.0
    longitude   (y, x) float64 265kB dask.array<chunksize=(157, 211), meta=np.ndarray>
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
  * time        (time) datetime64[ns] 60kB 2023-01-01 ... 2025-07-31T21:00:00
    valid_time  datetime64[ns] 8B ...
  * x           (x) int64 2kB 0 1 2 3 4 5 6 7 ... 204 205 206 207 208 209 210
  * y           (y) int64 1kB 0 1 2 3 4 5 6 7 ... 150 151 152 153 154 155 156
Pressure coord: level [1000.  950.  900.  850.  800.  750.  700.  600.  500.]
u ('time', 'level', 'y', 'x')
v ('time', 'level', 'y', 'x')
t ('time', 'level', 'y', 'x')
z ('time', 'level', 'y', 'x')
r ('time', 'level', 'y', 'x')


In [5]:
atmospheric_dims = ("time", "level","y", "x")
atmospheric_fields = [var for var in ds.data_vars if ds[var].dims == atmospheric_dims]
for var in atmospheric_fields:
    print(f" - {var}")

 - r
 - t
 - u
 - v
 - z


In [6]:
surface_dims = ("time", "y", "x")
surface_fields = [var for var in ds.data_vars if ds[var].dims == surface_dims]
for var in surface_fields:
    print(f" - {var}")

 - mcc
 - msl
 - power
 - synthetic_windpower
 - t2m
 - wdir10_cos
 - wdir10_sin
 - wdir_cos100
 - wdir_cos150
 - wdir_cos200
 - wdir_cos50
 - wdir_sin100
 - wdir_sin150
 - wdir_sin200
 - wdir_sin50
 - ws10
 - ws100
 - ws150
 - ws200
 - ws50


In [7]:
constant_dims = ("y","x")
constant_fields = [var for var in ds.data_vars if ds[var].dims == constant_dims]
for var in constant_fields:
    print(f" - {var}")

 - lsm
 - mask
 - surface_geopotential


In [8]:
recipe_keys = [
    "name",
    "description",
    "dates",
    "input",
    "build"
]
recipe = dict.fromkeys(recipe_keys)

In [9]:
recipe["name"] = "Cerra-BOZ-3h-v0"
recipe["description"] = "Toy dataset for wind power prediction in BOZ"

In [10]:
url = "Cerra_boz.zarr"

In [11]:
# Set the dates part of the config
dates = {
    "start": "2023-01-01T00:00:00",
    "end": "2025-07-31T21:00:00",
    "frequency": "3h"
}

# and add it to the config
recipe["dates"] = dates

In [12]:
surface_variables = {
    "xarray-zarr": {                    # The key defining the type, the value is another dictionary with options
        "url": url,                     # The URL of the dataset
        "param": surface_fields,    # The variables to extract                   
    }
}

In [13]:
pressure_levels=[500,600,700,750,800,850,900,950,1000]
upper_air_variables = {
    "xarray-zarr": {                    # The key defining the type, the value is another dictionary with options
        "url": url,                     # The URL of the dataset
        "param": atmospheric_fields,    # The variables to extract                   
    }
}

In [19]:
constant_variables = { 
    "repeated-dates": {      # The key defining the type.
        "mode": "constant",  # The mode, defining here that the fields are constant, other options (climatology, closest) exist
        "source": {          # Which source to use for the constant fields
            "xarray-zarr": { # An xarray-zarr source as defined above
                "url": url,
                "param": ["surface_geopotential", "lsm","mask"],
            }
        }
    }
}

In [20]:
forcings = {
    "forcings": {
        "template": "${input.join.1.xarray-zarr}", # Path to another source in the recipe dictionary.
        "param": [
            "cos_latitude",
            "cos_longitude",
            "sin_latitude",
            "sin_longitude",
            "julian_day",
            "insolation",
        ]
    }
}

In [21]:
# We can combine all these datasets by joining them using the join key in a dictionary
recipe["input"] = {
    "join": [
        constant_variables,
        upper_air_variables,
        surface_variables,
        forcings
    ]
}

In [22]:
build = {
    "group_by": "daily", 
    "variable_naming": "param_levelist",
}

recipe["build"] = build

In [23]:
#I ran this in terminal, not here, but the command is below
print(
    yaml.dump(
        recipe,
        default_flow_style=False
    )
)

build:
  group_by: daily
  variable_naming: param_levelist
dates:
  end: '2025-07-31T21:00:00'
  frequency: 3h
  start: '2023-01-01T00:00:00'
description: Toy dataset for wind power prediction in BOZ
input:
  join:
  - repeated-dates:
      mode: constant
      source:
        xarray-zarr:
          param:
          - surface_geopotential
          - lsm
          - mask
          url: Cerra_boz.zarr
  - xarray-zarr:
      param:
      - r
      - t
      - u
      - v
      - z
      url: Cerra_boz.zarr
  - xarray-zarr:
      param:
      - mcc
      - msl
      - power
      - synthetic_windpower
      - t2m
      - wdir10_cos
      - wdir10_sin
      - wdir_cos100
      - wdir_cos150
      - wdir_cos200
      - wdir_cos50
      - wdir_sin100
      - wdir_sin150
      - wdir_sin200
      - wdir_sin50
      - ws10
      - ws100
      - ws150
      - ws200
      - ws50
      url: Cerra_boz.zarr
  - forcings:
      param:
      - cos_latitude
      - cos_longitude
      - sin_latitude
 

In [17]:
name = recipe["name"]
recipe_path = f"{name}.yaml"
with open(recipe_path, 'w') as file:
    yaml.dump(recipe, file, default_flow_style=False)
dataset_path = "Cerra_boz_Anemoi.zarr"

In [ ]:
#!anemoi-datasets create --overwrite {recipe_path} {dataset_path}

In [2]:
!anemoi-datasets inspect Cerra_boz_Anemoi.zarr

📦 Path          : Cerra_boz_Anemoi.zarr
🔢 Format version: 0.30.0

📅 Start      : 2023-01-01 00:00
📅 End        : 2025-07-31 21:00
⏰ Frequency  : 3h
🚫 Missing    : 0
🌎 Resolution : None
🌎 Field shape: [157, 211]

📐 Shape      : 7,544 × 72 × 1 × 33,127 (67 GiB)
💽 Size       : 37.2 GiB (37.2 GiB)
📁 Files      : 7,583

   Index │ Variable             │         Min │      Max │       Mean │      Stdev
   ──────┼──────────────────────┼─────────────┼──────────┼────────────┼───────────
       0 │ cos_latitude         │    0.548867 │ 0.666868 │   0.606946 │  0.0310695
       1 │ cos_longitude        │    0.983806 │        1 │   0.995883 │ 0.00412394
       2 │ insolation           │           0 │ 0.905152 │   0.198574 │   0.263475
       3 │ julian_day           │           0 │  365.875 │    177.387 │    108.017
       4 │ lsm                  │           0 │        1 │   0.526356 │   0.487836
       5 │ mcc                  │           0 │      100 │    33.9572 │    38.0521
       6 │ msl     